In [3]:
#!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 13.9 MB/s  0:00:00.6 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] 1/2 [statsmodels]


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

# Read the data
try:
    df = pd.read_csv('bangkok-air-quality.csv')

    # Clean column names
    df.columns = df.columns.str.strip()

    # Ensure date is datetime
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').set_index('date')

    # Replace empty spaces or non-numeric with NaN in 'pm25'
    df['pm25'] = pd.to_numeric(df['pm25'], errors='coerce')

    # Filter data from 2016 onwards as per the project scope
    df = df[df.index.year >= 2016]

    # Handle Missing Values (Mean Imputation)
    mean_pm25 = df['pm25'].mean()
    df['pm25'] = df['pm25'].fillna(mean_pm25)

    # Remove Outliers (IQR method)
    Q1 = df['pm25'].quantile(0.25)
    Q3 = df['pm25'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean = df[(df['pm25'] >= lower) & (df['pm25'] <= upper)].copy()

    # Thai Seasons Function
    def get_season(month):
        if month in [3, 4, 5]: return 'Summer'
        elif month in [6, 7, 8, 9, 10]: return 'Rainy'
        else: return 'Winter'

    df_clean['month'] = df_clean.index.month
    df_clean['season'] = df_clean['month'].apply(get_season)

    # Statistical Modeling (Descriptive Stats)
    stats = df_clean.groupby('season')['pm25'].agg(['mean', 'max', 'std']).round(2)
    print("--- Seasonal Stats ---")
    print(stats)

    # Month analysis for Key Findings
    monthly_stats = df_clean.groupby('month')['pm25'].mean().round(2)
    worst_month = monthly_stats.idxmax()
    print(f"\nWorst month index: {worst_month}")

    # EDA: Time-Series Decomposition
    df_ts = df_clean['pm25'].resample('D').mean().interpolate(method='time')

    # Plot 1: Decomposition
    plt.figure(figsize=(12, 8))
    decomposition = seasonal_decompose(df_ts.dropna(), model='additive', period=365)
    fig = decomposition.plot()
    fig.set_size_inches(12, 8)
    plt.suptitle('PM2.5 Time-Series Decomposition (Bangkok 2016-Present)', fontsize=14)
    plt.tight_layout()
    plt.savefig('pm25_decomposition.png')
    plt.close('all')

    # Plot 2: Seasonal Boxplot
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='season', y='pm25', data=df_clean, order=['Winter', 'Summer', 'Rainy'], palette='Set2')
    plt.title('PM2.5 Distribution by Thai Seasons (2016-Present)')
    plt.ylabel('PM2.5 Concentration (AQI)')
    plt.xlabel('Season')
    plt.savefig('pm25_seasonal_boxplot.png')
    plt.close('all')

    # Plot 3: 10-year trend (Monthly Average)
    plt.figure(figsize=(14, 6))
    df_clean['pm25'].resample('M').mean().plot(color='firebrick', linewidth=2)
    plt.title('Monthly Average PM2.5 in Bangkok (2016-Present)')
    plt.ylabel('Average PM2.5 (AQI)')
    plt.xlabel('Year')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig('pm25_10yr_trend.png')
    plt.close('all')

    print("\nPlots generated successfully.")
except Exception as e:
    print(f"Error: {e}")
    print(f"Error: {e}")
    print(f"Error: {e}")
    print(f"Error: {e}")

--- Seasonal Stats ---
         mean    max    std
season                     
Rainy   62.40  149.0  17.15
Summer  80.13  149.0  20.96
Winter  99.23  150.0  24.22

Worst month index: 1

Plots generated successfully.
